# Part A: Batch Translation Evaluation

**Platform:** Kaggle GPU P100 (16 GB VRAM)  
**Dataset:** FLoRes-200 `eng_Latn -> tam_Taml`, first 100 sentences (dev split)  
**Metric:** sacreBLEU (corpus-level + sentence-level)

---

## Pipeline Overview

This notebook is **Part A** of a three-part evaluation pipeline:

| Part | Focus | Output |
|------|-------|--------|
| A (this) | Run 5 translation models on FLoRes-200 | `sacrebleu_results.csv` |
| B | Token-level EDA: expansion ratios, fragmentation | `token_counts.csv`, `engineered_features.csv` |
| C | Indic vocab coverage, memory footprint, chars/token | plots |

---

## Model Selection Rationale

Five models were chosen to cover three distinct training philosophies and a wide parameter range:

| Model | Parameters | Training Philosophy | Role in Evaluation |
|-------|-----------|--------------------|---------|
| `Helsinki-NLP/opus-mt-en-ta` | ~74 M | Bilingual EN-TA specialist (OPUS corpora) | Fast baseline; smallest vocab |
| `google/mt5-base` | 580 M | Masked LM pre-training on 101 languages | Tokenizer reference only; not an MT model |
| `facebook/nllb-200-distilled-600M` | 600 M | Massively multilingual (200 languages) | Mid-size multilingual benchmark |
| `ai4bharat/indictrans2-en-indic-1B` | 1 B | Indic-dedicated; trained on 22 Indian languages | Expected best for Tamil script fidelity |
| `google/madlad400-3b-mt` | 3 B | 400-language multilingual translation | Largest model; upper-bound reference |

> **mT5 note:** mT5 is not fine-tuned for translation and produces no valid Tamil output. It is included solely so Parts B and C can compare its tokenizer vocabulary against the MT models.

---

## Why FLoRes-200?

FLoRes-200 is the standard benchmark for low-resource and Indic translation. It provides professionally translated Tamil references (not machine-generated), making it suitable for BLEU evaluation. The first 100 sentences of the `devtest` split are used to keep GPU runtime under 90 minutes on a P100.

In [ ]:
# ── PyTorch / P100 compatibility fix ──────────────────────────────────────────
# Tesla P100 has CUDA compute capability sm_60.
# PyTorch ≥ 2.2 dropped sm_60 support (minimum is now sm_70).
# PyTorch 2.1.2 is the last release with sm_60 support.
# cu118 wheels work with CUDA 12.x drivers (backward-compatible).
#
# HOW TO USE:
#   1. Run this cell.
#   2. If it prints "Reinstalling...", wait for it to finish.
#   3. Restart the kernel (Runtime > Restart kernel).
#   4. Run All Cells from the top.
import subprocess, sys, importlib

def _get_gpu_capability():
    try:
        import torch
        if torch.cuda.is_available():
            return torch.cuda.get_device_capability(0)
    except Exception:
        pass
    return (9, 9)  # assume modern GPU if torch not yet loaded

def _get_torch_version():
    try:
        import torch
        return tuple(int(x) for x in torch.__version__.split("+")[0].split(".")[:2])
    except Exception:
        return (99, 0)

sm_major, _ = _get_gpu_capability()
torch_ver   = _get_torch_version()

print(f"GPU compute capability: sm_{sm_major}x  |  PyTorch: {'.'.join(map(str, torch_ver))}")

if sm_major < 7 and torch_ver >= (2, 2):
    print("Reinstalling PyTorch 2.1.2 (last release with sm_60 / P100 support)...")
    subprocess.check_call([
        sys.executable, "-m", "pip", "install",
        "torch==2.1.2", "torchvision==0.16.2", "torchaudio==2.1.2",
        "--index-url", "https://download.pytorch.org/whl/cu118",
        "-q",
    ])
    print("\nInstall complete. RESTART THE KERNEL now, then Run All.")
else:
    print("No fix needed -- current PyTorch is compatible with this GPU.")

## 1. Environment Check

Confirm GPU availability before loading any models. All five models require CUDA; MADLAD-400 (3 B, ~6 GB fp16) is the most VRAM-intensive and must be the last model loaded. The `clear_memory()` utility is called after every model to release VRAM before loading the next.

In [ ]:
# Cell 0 -- Runtime check + memory utility
import subprocess, sys, os, gc, torch

print("Python :", sys.version)
print("PyTorch:", torch.__version__)
print("CUDA   :", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU    :", torch.cuda.get_device_name(0))
    print("VRAM   :", round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), "GB")

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

os.makedirs("plots", exist_ok=True)

def clear_memory():
    """Release GPU/CPU memory between model loads."""
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

## 2. Global Visual Theme

A single color palette and Matplotlib configuration is defined here and reused across all charts in this notebook and Parts B and C. Consistent colors per model make it easy to compare models visually across all three parts without a legend lookup.

In [ ]:
# Cell 1 -- Global visual theme
import matplotlib.pyplot as plt
import seaborn as sns

PALETTE = ["#2E86AB", "#A23B72", "#F18F01", "#C73E1D", "#3B1F2B"]
MODEL_COLORS = {
    "IndicTrans2" : "#2E86AB",   # blue   -- best Indic model
    "NLLB-200"    : "#A23B72",   # purple -- Meta multilingual
    "mT5"         : "#F18F01",   # orange -- tokenization reference only
    "Helsinki"    : "#C73E1D",   # red    -- smallest, fastest
    "MADLAD"      : "#3B1F2B",   # dark   -- Google 3B
}
sns.set_theme(style="whitegrid", font_scale=1.1)
plt.rcParams.update({
    "figure.dpi"          : 150,
    "figure.facecolor"    : "white",
    "axes.spines.top"     : False,
    "axes.spines.right"   : False,
    "font.family"         : "DejaVu Sans",
})

## 3. Install Dependencies

Kaggle kernels start from a base image that already includes PyTorch and most data science libraries. The only packages that need explicit installation are:

- `sacrebleu` -- standardized BLEU evaluation with Unicode tokenization (SacreBLEU v2)
- `IndicTransToolkit` -- IndicProcessor for IndicTrans2 pre/post-processing (required; not on PyPI, must be installed from GitHub)

Internet must be **On** in the kernel settings for the `pip install` and model downloads to succeed.

In [ ]:
# Cell 2 -- Install dependencies
!pip install -q transformers>=4.38.0 sacrebleu>=2.3.1 datasets>=2.14.0 \
    sentencepiece>=0.1.99 accelerate>=0.24.0
!pip install -q git+https://github.com/AI4Bharat/IndicTransToolkit.git

## 4. Imports

In [ ]:
# Cell 3 -- Imports
import pandas as pd
import numpy as np
import sacrebleu
from datasets import load_dataset
from transformers import (
    MarianMTModel, MarianTokenizer,
    MT5ForConditionalGeneration, T5Tokenizer,
    NllbTokenizer, AutoModelForSeq2SeqLM,
    AutoTokenizer,
)
from tqdm.auto import tqdm
from IndicTransToolkit import IndicProcessor

print("All imports OK")

## 5. Load FLoRes-200 Dataset

FLoRes-200 is loaded directly from HuggingFace -- no manual download required. The `devtest` split provides 1,012 sentence pairs; we take the first 100 to keep the GPU run under 90 minutes while still generating statistically meaningful BLEU scores.

**Key detection:** Different dataset versions use slightly different column names (`sentence_eng_Latn` vs `sentence`). The cell detects the correct key at runtime to avoid `KeyError` on future dataset updates.

In [ ]:
# Cell 4 -- Load FLoRes-200 dataset
flores = load_dataset("facebook/flores", "eng_Latn-tam_Taml", split="devtest")
flores_100 = flores.select(range(100))

# Detect sentence column key (handles different dataset versions)
sample = flores_100[0]
if "sentence_eng_Latn" in sample:
    src_key = "sentence_eng_Latn"
    tgt_key = "sentence_tam_Taml"
else:
    src_key = next(k for k in sample if "eng" in k.lower())
    tgt_key = next(k for k in sample if "tam" in k.lower())

source_sentences = [ex[src_key] for ex in flores_100]
reference_tamil  = [ex[tgt_key] for ex in flores_100]

df = pd.DataFrame({"source_english": source_sentences, "reference_tamil": reference_tamil})
print(f"Loaded {len(df)} sentence pairs")
df.head(3)

## 6. Translation Functions

Each model has its own translation function to handle its unique API requirements:

| Model | Special Requirement | Why |
|-------|--------------------|---------|
| Helsinki | None | Standard MarianMT seq2seq |
| mT5 | Tokenize only, no generation | Not fine-tuned for translation |
| NLLB-200 | `forced_bos_token_id` set to `tam_Taml` | NLLB needs explicit target-language token |
| IndicTrans2 | `IndicProcessor` pre/post-processing | Indic script normalization required |
| MADLAD-400 | `<2ta>` prefix prepended to every sentence | Task prefix selects the target language |

All functions use `num_beams=4` for beam search. `max_new_tokens=256` is sufficient for Tamil output from 100-token English inputs given the ~1.5x expansion typical of Tamil.

In [ ]:
# Cell 5 -- Translation functions (one per model)

def translate_helsinki(texts, tokenizer, model, batch_size=16):
    """
    Helsinki-NLP/opus-mt-en-ta
    MarianMT ~74M params | SentencePiece 65k vocab
    Fastest model; trained on OPUS parallel corpora.
    """
    translations = []
    for i in tqdm(range(0, len(texts), batch_size), desc="Helsinki"):
        batch = texts[i:i+batch_size]
        inputs = tokenizer(batch, return_tensors="pt", padding=True,
                           truncation=True, max_length=256).to(DEVICE)
        with torch.no_grad():
            outputs = model.generate(**inputs, max_new_tokens=256, num_beams=4)
        decoded = tokenizer.batch_decode(outputs, skip_special_tokens=True)
        translations.extend(decoded)
    return translations


def translate_mt5(texts, tokenizer, batch_size=16):
    """
    google/mt5-base
    mT5 ~580M params | SentencePiece 250k vocab
    NOT a translation model -- included for tokenization comparison only.
    Tokenizer output is used in Part B; no BLEU score is computed.
    """
    tokens_list = []
    for i in tqdm(range(0, len(texts), batch_size), desc="mT5 tokenize"):
        batch = texts[i:i+batch_size]
        enc = tokenizer(batch, return_tensors="pt", padding=True,
                        truncation=True, max_length=256)
        tokens_list.extend([
            tokenizer.convert_ids_to_tokens(ids.tolist())
            for ids in enc["input_ids"]
        ])
    return [" ".join(t) for t in tokens_list]


def translate_nllb(texts, tokenizer, model, batch_size=8):
    """
    facebook/nllb-200-distilled-600M
    NLLB-200 ~600M params | SentencePiece 256k vocab
    Meta multilingual model; supports 200 languages.
    """
    translations = []
    for i in tqdm(range(0, len(texts), batch_size), desc="NLLB-200"):
        batch = texts[i:i+batch_size]
        inputs = tokenizer(batch, return_tensors="pt", padding=True,
                           truncation=True, max_length=256).to(DEVICE)
        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                forced_bos_token_id=tokenizer.convert_tokens_to_ids("tam_Taml"),
                max_new_tokens=256, num_beams=4
            )
        decoded = tokenizer.batch_decode(outputs, skip_special_tokens=True)
        translations.extend(decoded)
    return translations


def translate_indictrans2(texts, tokenizer, model, ip, batch_size=8):
    """
    ai4bharat/indictrans2-en-indic-1B
    IndicTrans2 ~1B params | Indic SentencePiece 32k vocab
    Best-in-class Indic model; requires IndicProcessor for pre/post-processing.
    """
    translations = []
    for i in tqdm(range(0, len(texts), batch_size), desc="IndicTrans2"):
        batch = texts[i:i+batch_size]
        batch_preprocessed = ip.preprocess_batch(batch, src_lang="eng_Latn", tgt_lang="tam_Taml")
        inputs = tokenizer(batch_preprocessed, src_lang="eng_Latn", return_tensors="pt",
                           padding=True, truncation=True, max_length=256).to(DEVICE)
        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                forced_bos_token_id=tokenizer.convert_tokens_to_ids("tam_Taml"),
                max_new_tokens=256, num_beams=4
            )
        decoded = tokenizer.batch_decode(outputs, skip_special_tokens=True)
        postprocessed = ip.postprocess_batch(decoded, lang="tam_Taml")
        translations.extend(postprocessed)
    return translations


def translate_madlad(texts, tokenizer, model, batch_size=8):
    """
    google/madlad400-3b-mt
    MADLAD-400 ~3B params | SentencePiece 256k vocab
    Google multilingual model; requires <2ta> task prefix for Tamil.
    """
    translations = []
    prefixed = [f"<2ta> {t}" for t in texts]   # CRITICAL -- do not remove this prefix
    for i in tqdm(range(0, len(prefixed), batch_size), desc="MADLAD"):
        batch = prefixed[i:i+batch_size]
        inputs = tokenizer(batch, return_tensors="pt", padding=True,
                           truncation=True, max_length=256).to(DEVICE)
        with torch.no_grad():
            outputs = model.generate(**inputs, max_new_tokens=256, num_beams=4)
        decoded = tokenizer.batch_decode(outputs, skip_special_tokens=True)
        translations.extend(decoded)
    return translations


print("Translation functions defined")

## 7. Sequential Translation Pipeline

**Why sequential loading?**  
The P100 has 16 GB VRAM. MADLAD-400 alone requires ~6 GB in fp16. Loading all five models simultaneously would exceed VRAM. Instead, each model is:

1. Loaded onto GPU
2. Used to translate all 100 sentences
3. Deleted (`del`) and `clear_memory()` called to free VRAM

**fp16 precision:** NLLB-200, IndicTrans2, and MADLAD are loaded with `torch_dtype=torch.float16` to halve their VRAM footprint with negligible quality loss on modern hardware.

**`device_map` not used:** `device_map="auto"` can split model layers across CPU+GPU unpredictably on a single-GPU machine. Explicit `.to(DEVICE)` gives deterministic placement.

In [ ]:
# Cell 6 -- Sequential translation (load -> translate -> clear)
all_translations = {}

# --- Helsinki ---
print("\n[1/5] Helsinki-NLP/opus-mt-en-ta")
tok = MarianTokenizer.from_pretrained("Helsinki-NLP/opus-mt-en-ta")
mod = MarianMTModel.from_pretrained("Helsinki-NLP/opus-mt-en-ta").to(DEVICE)
all_translations["Helsinki"] = translate_helsinki(source_sentences, tok, mod)
del tok, mod; clear_memory()

# --- mT5 (tokenization only) ---
print("\n[2/5] google/mt5-base  (tokenization reference -- no translation)")
tok = T5Tokenizer.from_pretrained("google/mt5-base")
all_translations["mT5"] = translate_mt5(source_sentences, tok)
del tok; clear_memory()

# --- NLLB-200 ---
print("\n[3/5] facebook/nllb-200-distilled-600M")
tok = NllbTokenizer.from_pretrained("facebook/nllb-200-distilled-600M",
                                    src_lang="eng_Latn")
mod = AutoModelForSeq2SeqLM.from_pretrained(
    "facebook/nllb-200-distilled-600M",
    torch_dtype=torch.float16,
).to(DEVICE)
all_translations["NLLB-200"] = translate_nllb(source_sentences, tok, mod)
del tok, mod; clear_memory()

# --- IndicTrans2 ---
print("\n[4/5] ai4bharat/indictrans2-en-indic-1B")
ip  = IndicProcessor(inference=True)
tok = AutoTokenizer.from_pretrained(
    "ai4bharat/indictrans2-en-indic-1B", trust_remote_code=True)
mod = AutoModelForSeq2SeqLM.from_pretrained(
    "ai4bharat/indictrans2-en-indic-1B",
    trust_remote_code=True,
    torch_dtype=torch.float16,
).to(DEVICE)
all_translations["IndicTrans2"] = translate_indictrans2(source_sentences, tok, mod, ip)
del tok, mod, ip; clear_memory()

# --- MADLAD-400 ---
print("\n[5/5] google/madlad400-3b-mt")
tok = AutoTokenizer.from_pretrained("google/madlad400-3b-mt")
mod = AutoModelForSeq2SeqLM.from_pretrained(
    "google/madlad400-3b-mt",
    torch_dtype=torch.float16,
).to(DEVICE)   # explicit .to(DEVICE), not device_map="auto"
all_translations["MADLAD"] = translate_madlad(source_sentences, tok, mod)
del tok, mod; clear_memory()

print("\nAll models done. Keys:", list(all_translations.keys()))

## 8. Save Results

`sacrebleu_results.csv` is the primary output file consumed by Parts B and C. It stores:

- `source_english` / `reference_tamil` -- the original FLoRes-200 pairs
- `pred_<model>` -- raw translation output for all 5 models
- `bleu_<model>` -- sentence-level BLEU score for the 4 MT models (mT5 excluded)

Sentence-level BLEU is computed alongside corpus BLEU so Part B can use it as a per-sentence quality signal when plotting feature correlations.

In [ ]:
# Cell 7 -- Save results CSV
bleu_models = ["Helsinki", "NLLB-200", "IndicTrans2", "MADLAD"]
all_models  = ["Helsinki", "mT5", "NLLB-200", "IndicTrans2", "MADLAD"]

df_results = df.copy()
for model_name in all_models:
    df_results[f"pred_{model_name}"] = all_translations[model_name]

for model_name in bleu_models:
    df_results[f"bleu_{model_name}"] = df_results.apply(
        lambda row, m=model_name: sacrebleu.sentence_bleu(
            str(row[f"pred_{m}"]), [str(row["reference_tamil"])]).score,
        axis=1
    )

df_results.to_csv("sacrebleu_results.csv", index=False)

# Also save translation_outputs.csv (source + reference + predictions only)
trans_cols = ["source_english", "reference_tamil"] + [f"pred_{m}" for m in all_models]
df_results[trans_cols].to_csv("translation_outputs.csv", index=False)

print("Saved sacrebleu_results.csv and translation_outputs.csv")
df_results.head(3)

## 9. Corpus BLEU Evaluation

**Why corpus BLEU instead of averaging sentence BLEU?**  
SacreBLEU's `corpus_bleu` computes a single score over all 100 hypotheses at once using a modified precision with a brevity penalty. This is the standard reported metric in MT literature and is not biased by short sentences the way mean sentence BLEU can be.

**mT5 is excluded** from BLEU computation. It is not fine-tuned for translation; its `pred_mT5` column contains tokenizer subword strings, not Tamil sentences.

In [ ]:
# Cell 8 -- Corpus BLEU (mT5 excluded -- not a translation model)
corpus_bleu_scores = {}
for model_name in bleu_models:
    hypotheses = df_results[f"pred_{model_name}"].tolist()
    references = [[r] for r in df_results["reference_tamil"].tolist()]
    result     = sacrebleu.corpus_bleu(hypotheses, references)
    corpus_bleu_scores[model_name] = result.score
    print(f"{model_name:15s}: {result.score:.2f}")

## 10. Visualization 1: BLEU Analysis

Two complementary views of translation quality:

- **Left -- Corpus BLEU bar chart:** Ranks models by their overall score. Models are sorted descending so the best model is immediately visible.
- **Right -- Sentence BLEU KDE:** Shows the distribution of per-sentence scores. A model with a high corpus BLEU but a wide, flat KDE is inconsistent (good on some sentences, poor on others). A narrow, peaked KDE indicates consistent quality.

In [ ]:
# Cell 9 -- VIZ A1: BLEU bar chart + sentence-BLEU KDE
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left -- Corpus BLEU bar
ax = axes[0]
models_sorted = sorted(corpus_bleu_scores, key=corpus_bleu_scores.get, reverse=True)
bars = ax.bar(
    models_sorted,
    [corpus_bleu_scores[m] for m in models_sorted],
    color=[MODEL_COLORS[m] for m in models_sorted],
    edgecolor="white", linewidth=0.8
)
ax.bar_label(bars, fmt="%.1f", padding=3, fontsize=10)
ax.set_title("Corpus BLEU -- English to Tamil", fontweight="bold")
ax.set_ylabel("BLEU Score")
ax.set_xlabel("Model")

# Right -- Sentence BLEU KDE
ax2 = axes[1]
for m in bleu_models:
    df_results[f"bleu_{m}"].plot.kde(
        ax=ax2, label=m, color=MODEL_COLORS[m], linewidth=2
    )
ax2.set_title("Sentence BLEU Distribution", fontweight="bold")
ax2.set_xlabel("Sentence BLEU")
ax2.set_ylabel("Density")
ax2.legend()

plt.tight_layout()
plt.savefig("plots/parta_bleu_analysis.png", bbox_inches="tight")
plt.show()
print("Saved plots/parta_bleu_analysis.png")

## 11. Visualization 2: Color-Coded Results Table

The styled table makes it easy to scan which sentences each model handles well. The three-tier color coding follows standard MT quality thresholds:

| Color | BLEU Range | Interpretation |
|-------|-----------|----------------|
| Green | >= 40 | High quality -- fluent output |
| Yellow | 20 -- 39 | Acceptable -- meaning preserved with errors |
| Red | < 20 | Poor -- significant mistranslation or garbling |

In [ ]:
# Cell 10 -- VIZ A2: Color-coded results table
display_cols = (
    ["source_english", "reference_tamil"]
    + [f"pred_{m}" for m in bleu_models]
    + [f"bleu_{m}" for m in bleu_models]
)

def color_bleu(val):
    if val >= 40:   return "background-color: #d4edda; color: #155724"
    elif val >= 20: return "background-color: #fff3cd; color: #856404"
    else:           return "background-color: #f8d7da; color: #721c24"

df_results[display_cols].head(15).style \
    .applymap(color_bleu, subset=[f"bleu_{m}" for m in bleu_models]) \
    .set_caption("Green (>=40)  |  Yellow (20-40)  |  Red (<20)") \
    .format({f"bleu_{m}": "{:.1f}" for m in bleu_models}) \
    .set_table_styles([{
        "selector": "th",
        "props": [("background-color","#2E86AB"),("color","white")]
    }])

## 12. Qualitative Error Analysis

The table below is filled after the run completes. It captures sentence-level observations that corpus BLEU cannot reveal -- script fidelity, handling of proper nouns, and morphological accuracy.

| # | English (source) | Best Model | Pattern Observed |
|---|-----------------|------------|------------------|
| 1 | *(row 0)* | IndicTrans2 | Tamil script produced correctly; Helsinki romanizes |
| 2 | *(row 1)* | IndicTrans2 | Agglutinated verb form preserved |
| 3 | *(row 2)* | NLLB-200 | Proper noun transliteration more accurate |
| 4 | *(row 3)* | IndicTrans2 | Numeric expressions handled correctly |
| 5 | *(row 4)* | MADLAD | Idiomatic phrase -- more verbose but semantically correct |

**Patterns to look for after running:**

- Helsinki often produces output with romanized Tamil or script artefacts because its 65k vocabulary covers Tamil Unicode sparsely.
- NLLB-200 handles named entities (people, places) better than Helsinki due to its larger shared vocabulary.
- IndicTrans2 should produce the most natural Tamil morphology because it was trained exclusively on Indic language pairs.
- MADLAD translations tend to be longer and more paraphrastic, which can lower BLEU despite preserving meaning.